In [1]:
from ipyparallel import Cluster
c = await Cluster(engines="mpi").start_and_connect(n=4, activate=True)

Starting 4 engines with <class 'ipyparallel.cluster.launcher.MPIEngineSetLauncher'>


  0%|          | 0/4 [00:00<?, ?engine/s]

In [4]:
%%px
from ngsolve import *
from netgen.occ import *
from ngsolve.krylovspace import BramblePasciakCG
from mpi4py.MPI import COMM_WORLD as comm

box = Box((0,0,0), (2,0.41,0.41))
box.faces.name="wall"
box.faces.Min(X).name="inlet"
box.faces.Max(X).name="outlet"
cyl = Cylinder((0.2,0,0.2), Y, h=0.41,r=0.05)
cyl.faces.name="cyl"
shape = box-cyl
ngmesh = OCCGeometry(shape).GenerateMesh(maxh=0.05, comm=comm)
    
for r in range(1): ngmesh.Refine()
mesh = Mesh(ngmesh)
print (mesh.GetNE(VOL))

[stdout:0] 0


[stdout:1] 35760


[stdout:3] 37208


[stdout:2] 35768


In [5]:
%%px
import ngsolve.ngs2petsc as n2p
import petsc4py.PETSc as psc

In [6]:
%%px
V = VectorH1(mesh, order=1, dirichlet="wall|inlet|cyl")
V1 = H1(mesh, order=1, dirichlet="wall|inlet|cyl")
Q = H1(mesh, order=1)
printonce ("ndof = ", V.ndofglobal,'+',Q.ndofglobal,'=',
        V.ndofglobal+Q.ndofglobal)

u,v = V.TnT()
u1,v1 = V1.TnT()
p,q = Q.TnT()

h = specialcf.mesh_size

bfa1 = BilinearForm(InnerProduct(grad(u1),grad(v1))*dx)
bfb = BilinearForm(div(u)*q*dx).Assemble()
bfc = BilinearForm(h*h*grad(p)*grad(q)*dx).Assemble()

prea1 = Preconditioner(bfa1, "gamg") # AMG precond from PETSc
bfa1.Assemble()

# make block-diagonal A matrix:
mata = sum( [Ri.T@bfa1.mat@Ri for Ri in V.restrictions] )
prea = sum( [Ei@prea1@Ei.T for Ei in V.embeddings])    
    
bfschur = BilinearForm(p*q*dx, diagonal=True).Assemble()
preschur = bfschur.mat.Inverse()

[stdout:0] ndof =  66090 + 22030 = 88120


In [7]:
%%px
gfu = GridFunction(V)
gfp = GridFunction(Q)

uin = (1.5*4*y*(0.41-y)/(0.41*0.41)*z*(0.41-z)/0.41**2,0, 0)

gfu.Set(uin, definedon=mesh.Boundaries("inlet"))

resf = (-mata * gfu.vec).Evaluate()
resg = (-bfb.mat * gfu.vec).Evaluate()

sol = BramblePasciakCG (A=mata, B=bfb.mat, C=bfc.mat, f=resf, g=resg, \
                preA=prea, preS=preschur, maxit=500, 
                printrates='\r' if comm.rank==0 else False)

gfu.vec.data += sol[0]
gfp.vec.data += sol[1]

[stdout:0] lammin/lammax =  0.5052418810055732 / 0.9982329720240131


In [8]:
gfu = c[:]["gfu"]

In [9]:
from ngsolve import *
from ngsolve.webgui import Draw
ea = { "euler_angles" : (-77, 6, 47) }
clipping = { "clipping" : {"y":1, "z":0, "function" : True } }
Draw (Norm(gfu[0]), gfu[0].space.mesh, **ea, **clipping, order=1)
Draw (Norm(gfu[2]), gfu[2].space.mesh, **ea, **clipping, order=1);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (-…

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (-…

In [10]:
gfp = c[:]["gfp"]
Draw (gfp[0], order=1, **ea, **clipping);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (-…

In [11]:
c.shutdown(hub=True)

Controller stopped: {'exit_code': 0, 'pid': 160021, 'identifier': 'ipcontroller-1775207052-36i9-159986'}
Output for ipengine-1775207052-36i9-1775207053-159986:
      4 from mpi4py import MPI
      5 import numpy as np

ModuleNotFoundError: No module named 'petsc4py'
2026-04-03 16:04:26.283 [IPEngine.1.1] Exception in execute request:
---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
Cell In[2], line 1
----> 1 import ngsolve.ngs2petsc as n2p
      2 import petsc4py.PETSc as psc

File ~/Workspace/biodynopt/.venv/lib/python3.12/site-packages/ngsolve/ngs2petsc.py:3
      1 import ngsolve as ngs
      2 import netgen.meshing as ngm
----> 3 import petsc4py.PETSc as psc
      4 from mpi4py import MPI
      5 import numpy as np

ModuleNotFoundError: No module named 'petsc4py'
2026-04-03 16:04:26.283 [IPEngine.0.0] Aborting queue
2026-04-03 16:04:26.283 [IPEngine.2.2] Exception in execute reque